Calculates stats for known vs unknown BGC expression.

In [5]:
import pandas as pd

In [6]:
file_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_expression_antismash_mibig_refined.xlsx'
mibig_df = pd.read_excel(file_path)

In [7]:
mibig_df.head()

,Strain,BGC_ID,bgc_type,media_expressed,Most similar known cluster,Type MIBiG,Similarity
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,NRPS-like_nucleoside_other,not_expressed,mitomycin,Other:Aminocoumarin,23%
1,NBC_00001,11_furan_NBC_00001,furan,"ISP2, gluc, gly, malt",NaN,NaN,NaN
2,NBC_00001,12_NI-siderophore_NBC_00001,NI-siderophore,not_expressed,NaN,NaN,NaN
3,NBC_00001,13_nucleoside_NBC_00001,nucleoside,ISP2,NaN,NaN,NaN
4,NBC_00001,14_terpene_NBC_00001,terpene,SoyM,albaflavenone,Terpene,100%


In [8]:
df = mibig_df.copy()

# Convert Similarity to numeric
df["Similarity_num"] = (
    df["Similarity"]
    .replace("None", None)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Create known vs unknown
df["known_vs_unknown"] = df["Similarity_num"].apply(
    lambda x: "known" if pd.notna(x) and x >= 80 else "unknown"
)

In [9]:
# Create simple expressed flag
df["is_expressed"] = df["media_expressed"] != "not_expressed"

# Count
summary = pd.crosstab(df["known_vs_unknown"], df["is_expressed"])
print(summary)

is_expressed      False  True 
known_vs_unknown              
known               405    641
unknown            1919   1028


In [10]:
# Count number of media conditions per row
df["n_conditions"] = df["media_expressed"].apply(
    lambda x: 0 if x == "not_expressed" else len([i.strip() for i in x.split(",")])
)

In [12]:
df[df["is_expressed"]].groupby("known_vs_unknown")["n_conditions"].mean()

known_vs_unknown
known      4.196568
unknown    3.406615
Name: n_conditions, dtype: float64

In [13]:
save_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_expression_antismash_mibig_refined.xlsx'
df.to_excel(save_path)

In [26]:
file_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_average_core_expression_per_condition.csv'
bgc_expression = pd.read_csv(file_path)

In [27]:
bgc_expression

,Strain,Medium,Sample,BGC_ID,Avg_Core_Expr
0,NBC_01734,DNPM,1734_DNPM_2,1_terpene_NBC_01734,2.175151
1,NBC_01734,DNPM,1734_DNPM_2,10_terpene_NBC_01734,2.820813
2,NBC_01734,DNPM,1734_DNPM_2,11_LAP_thioamitides_thiopeptide_NBC_01734,1.439695
3,NBC_01734,DNPM,1734_DNPM_2,12_NI-siderophore_NBC_01734,2.967205
4,NBC_01734,DNPM,1734_DNPM_2,13_terpene_NBC_01734,3.844478
...,...,...,...,...,...
42213,NBC_01770,malt,1770_malt_1,5_ectoine_NBC_01770,5.141578
42214,NBC_01770,malt,1770_malt_1,6_T3PKS_NBC_01770,3.349465
42215,NBC_01770,malt,1770_malt_1,7_melanin_NBC_01770,3.640247
42216,NBC_01770,malt,1770_malt_1,8_LAP_lanthipeptide-class-v_NBC_01770,0.709731


In [28]:
# --- 1. Start from your expression dataframe ---
df_expr = bgc_expression.copy()

# --- 2. Pivot: long → wide (one column per medium) ---
wide_df = df_expr.pivot_table(
    index=["Strain", "BGC_ID"],
    columns="Medium",
    values="Avg_Core_Expr",
    aggfunc="mean"
).reset_index()

# --- 3. Clean column names ---
wide_df.columns.name = None

wide_df = wide_df.rename(columns={
    col: f"{col}_expr" for col in wide_df.columns if col not in ["Strain", "BGC_ID"]
})

# --- 4. Fill missing values (optional but recommended) ---
wide_df = wide_df.fillna(0)

# --- 5. Add expression metrics ---
expr_cols = [c for c in wide_df.columns if c.endswith("_expr")]


# max expression across conditions
wide_df["max_expression"] = wide_df[expr_cols].max(axis=1)

# mean expression across conditions
wide_df["mean_expression"] = wide_df[expr_cols].mean(axis=1)


In [29]:
wide_df

,Strain,BGC_ID,DNPM_expr,ISP2_expr,MA_expr,SoyM_expr,TSB_expr,gluc_expr,gly_expr,malt_expr,max_expression,mean_expression
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,2.052220,2.800726,2.450850,2.577224,1.947189,1.878496,2.492139,1.864145,2.800726,2.257874
1,NBC_00001,11_furan_NBC_00001,4.652892,8.989872,4.523973,4.335328,1.313786,9.123664,11.350178,6.424600,11.350178,6.339287
2,NBC_00001,12_NI-siderophore_NBC_00001,4.085232,4.739515,4.936622,4.533782,3.174355,3.652043,3.466901,4.379999,4.936622,4.121056
3,NBC_00001,13_nucleoside_NBC_00001,4.913477,6.202608,4.793508,5.562214,4.056903,4.475249,3.412159,4.636625,6.202608,4.756593
4,NBC_00001,14_terpene_NBC_00001,4.730438,5.470210,5.117656,5.943101,2.672705,3.366341,3.865667,3.898380,5.943101,4.383062
...,...,...,...,...,...,...,...,...,...,...,...,...
3988,NBC_01815,5_NI-siderophore_NBC_01815,4.250500,4.010117,5.768043,6.256433,2.747685,7.395006,8.911628,9.718436,9.718436,6.132231
3989,NBC_01815,6_T2PKS_NBC_01815,2.007454,1.350837,1.396239,1.323844,1.665701,1.867437,1.489366,1.353242,2.007454,1.556765
3990,NBC_01815,7_RiPP-like_NBC_01815,7.023913,6.111471,4.933062,10.342943,6.122886,6.047555,5.546558,5.899081,10.342943,6.503434
3991,NBC_01815,8_terpene_NBC_01815,7.723913,7.241013,7.262236,6.308092,8.067368,6.647379,4.835763,6.540297,8.067368,6.828258


In [33]:
save_path = '/Users/annasve/Desktop/article_data/output/expression_overall/core_gene_expression_wide.xlsx'
wide_df.to_excel(save_path)

In [30]:
combined_df = pd.merge(df, wide_df, on = 'BGC_ID', how = 'left')

In [31]:
combined_df

,Strain_x,BGC_ID,bgc_type,media_expressed,Most similar known cluster,Type MIBiG,Similarity,Similarity_num,known_vs_unknown,is_expressed,...,DNPM_expr,ISP2_expr,MA_expr,SoyM_expr,TSB_expr,gluc_expr,gly_expr,malt_expr,max_expression,mean_expression
0,NBC_00001,10_NRPS-like_nucleoside_other_NBC_00001,NRPS-like_nucleoside_other,not_expressed,mitomycin,Other:Aminocoumarin,23%,23.0,unknown,False,...,2.052220,2.800726,2.450850,2.577224,1.947189,1.878496,2.492139,1.864145,2.800726,2.257874
1,NBC_00001,11_furan_NBC_00001,furan,"ISP2, gluc, gly, malt",NaN,NaN,NaN,NaN,unknown,True,...,4.652892,8.989872,4.523973,4.335328,1.313786,9.123664,11.350178,6.424600,11.350178,6.339287
2,NBC_00001,12_NI-siderophore_NBC_00001,NI-siderophore,not_expressed,NaN,NaN,NaN,NaN,unknown,False,...,4.085232,4.739515,4.936622,4.533782,3.174355,3.652043,3.466901,4.379999,4.936622,4.121056
3,NBC_00001,13_nucleoside_NBC_00001,nucleoside,ISP2,NaN,NaN,NaN,NaN,unknown,True,...,4.913477,6.202608,4.793508,5.562214,4.056903,4.475249,3.412159,4.636625,6.202608,4.756593
4,NBC_00001,14_terpene_NBC_00001,terpene,SoyM,albaflavenone,Terpene,100%,100.0,known,True,...,4.730438,5.470210,5.117656,5.943101,2.672705,3.366341,3.865667,3.898380,5.943101,4.383062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3988,NBC_01815,5_NI-siderophore_NBC_01815,NI-siderophore,"gluc, gly, malt",desferrioxamin B/desferrioxamine E,Other,83%,83.0,known,True,...,4.250500,4.010117,5.768043,6.256433,2.747685,7.395006,8.911628,9.718436,9.718436,6.132231
3989,NBC_01815,6_T2PKS_NBC_01815,T2PKS,not_expressed,spore pigment,Polyketide,83%,83.0,known,False,...,2.007454,1.350837,1.396239,1.323844,1.665701,1.867437,1.489366,1.353242,2.007454,1.556765
3990,NBC_01815,7_RiPP-like_NBC_01815,RiPP-like,"DNPM, SoyM, TSB",NaN,NaN,NaN,NaN,unknown,True,...,7.023913,6.111471,4.933062,10.342943,6.122886,6.047555,5.546558,5.899081,10.342943,6.503434
3991,NBC_01815,8_terpene_NBC_01815,terpene,"DNPM, ISP2, MA, TSB, gluc",albaflavenone,Terpene,100%,100.0,known,True,...,7.723913,7.241013,7.262236,6.308092,8.067368,6.647379,4.835763,6.540297,8.067368,6.828258


In [32]:
save_path = '/Users/annasve/Desktop/article_data/output/expression_overall/bgc_expression_antismash_mibig_refined.xlsx'
combined_df.to_excel(save_path)